# Agent Repair: Controlled ICLR Experiments

**Inference and evaluation, not training or fine-tuning.** This notebook uses
the corrected local code snapshot, not the older GitHub notebook pipeline.
Upload this notebook and `agent-repair-iclr2027-code.zip` into the same Jupyter
folder. No repository push is needed. Existing runs are never deleted.

**$119 AWS allocation:** one 80GB-class GPU, one model, one token budget, four
conditions, three seeds, no all-origin sweep or extra offset diagnostics.
Keep the three QA datasets and run them sequentially. Extra datasets, model
families and the 72B judge are deferred. This is a reduced study, not evidence
that the original full experimental plan will finish within $119. Keep
$20 in reserve and at most $99 for compute. Do not add the cash fallback
budget to this allocation or count credit-covered usage as zero cost.

Recommended first run: Linux x86-64, Python 3.10-3.13,
persistent storage with at least 100 GB free (200 GB allocated is a starting
point, not a guarantee for every study). Run one dataset per GPU process.
A dedicated instance is more predictable than an availability-limited Colab
runtime. No cloud instance is provisioned by this notebook.

**Start with `EXECUTE_GPU = False` to check the package.** Then select the GPU
runtime, confirm persistent storage, set it to `True`, and run cells in order.
GPU rental continues to accrue while an instance is idle. Neither the budget
worksheet nor the repair-count guard enforces a provider spending limit.
AWS credits are not a spending cap: usage beyond eligible credits and taxes
can reach your payment method. Verify credit eligibility, remaining balance,
regional GPU quota and an independent stop before execution. AWS Budgets
alerts can lag; they are not a hard stop. Back up results before stopping.

The defaults are a development pilot, not publishable confirmatory results.
Human annotation and the close-method/full-cost comparison are separate work.

In [1]:
from pathlib import Path
import os

EXECUTE_GPU = False
PLATFORM = "aws"  # aws, runpod, colab, or other Linux Jupyter instance
DATASET = "hotpotqa"  # hotpotqa, musique, 2wikimultihopqa
PHASE = "pilot"       # pilot, development, test
RUN_ID = "aws119-pilot-v1"  # new ID for changed study settings
POOL_SIZE = 10        # initial questions, NOT number of failed trajectories
BATCH_SIZE = 2        # benchmark 2/4/8 on separate pilots before freezing
STRATEGY = "unc__perplexity__argmax__bt2"  # pilot candidate, not a selected winner
ORIGIN_SWEEP = False
INCLUDE_DIAGNOSTICS = False  # four core conditions only
MULTIPLIERS = [1.0]
MAX_REPAIR_EXECUTIONS = 120  # 10 questions * 4 conditions * 3 seeds, before reuse
GPU_INDEX = "0"

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
MODEL_REVISION = None  # first pilot resolves a commit; copy it to every later run
TEST_IDS = None        # absolute path to a JSON list, only for PHASE="test"
EXPLORED_IDS = None    # every previously explored/pilot/development question ID
FROZEN_POLICY = None   # completed study-policy JSON, frozen BEFORE viewing test outcomes

CODE_BUNDLE = Path("agent-repair-iclr2027-code.zip").resolve()
STORAGE_ROOT = Path("/workspace/agent-repair-iclr2027")
if PLATFORM == "colab":
    STORAGE_ROOT = Path("/content/drive/MyDrive/agent-repair-iclr2027")
# Only set True after checking the provider's persistent volume/mount.
PERSISTENT_STORAGE_CONFIRMED = False
TOTAL_BUDGET_USD = 119.0
NONCOMPUTE_RESERVE_USD = 20.0  # storage and contingency; tax may be charged separately
SPENT_SO_FAR_USD = 0.0        # gross usage since allocating $119, BEFORE credit offsets
SESSION_ALLOWANCE_USD = 10.0 # allocation only, NOT an automatic shutdown
GPU_HOURLY_USD = None        # enter the live EC2 On-Demand quote, not Capacity Blocks
BILLING_RATE_CONFIRMED = False
AWS_CREDITS_CONFIRMED = False  # current eligible balance, expiry, coverage and other usage
PROVIDER_BUDGET_CONTROLS_CONFIRMED = False  # independent stop; alerts alone are insufficient
PLANNED_FAILED_QUESTIONS = None  # use development variance to set the study size

import json as _parameter_json
globals().update(_parameter_json.loads('{"EXECUTE_GPU": true, "PLATFORM": "aws", "RUN_ID": "aws119-pilot-g7e-v2", "GPU_HOURLY_USD": 5.84531, "SPENT_SO_FAR_USD": 5.5, "PERSISTENT_STORAGE_CONFIRMED": true, "BILLING_RATE_CONFIRMED": true, "AWS_CREDITS_CONFIRMED": true, "PROVIDER_BUDGET_CONTROLS_CONFIRMED": true, "SESSION_ALLOWANCE_USD": 4.0}'))


## 1. Verify and Extract the Corrected Code

For Colab, upload both files before this cell and select a GPU runtime.
Drive mounting only occurs when `EXECUTE_GPU` is enabled. Store results on
persistent storage, not the runtime's temporary disk. On Runpod, verify the
volume attached at `/workspace`; a directory name alone does not prove persistence.
On AWS use EBS-backed storage, not the instance's ephemeral NVMe disk.

In [2]:
import hashlib
import json
import sys
import zipfile

EXPECTED_CODE_SHA256 = "1a48d14f60f807b1bd980201b84f5c86aacdb876d72bc7dcbcf8e0e26e2f6ab8"
if PLATFORM == "colab" and EXECUTE_GPU:
    from google.colab import drive
    drive.mount("/content/drive")
# A plan-only check stays in the current directory and does not mount cloud storage.
SOURCE_ROOT = STORAGE_ROOT if EXECUTE_GPU else Path.cwd() / ".iclr-notebook-check"
REPO = SOURCE_ROOT / "code" / EXPECTED_CODE_SHA256[:12]
if not CODE_BUNDLE.is_file():
    raise FileNotFoundError(f"Upload the matching code bundle: {CODE_BUNDLE}")
with zipfile.ZipFile(CODE_BUNDLE) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json"))
    hashes = manifest["files"]
    actual = hashlib.sha256(json.dumps(hashes, sort_keys=True).encode()).hexdigest()
    if actual != EXPECTED_CODE_SHA256:
        raise ValueError("Notebook and code bundle do not match; use the pair built together")
    if len(archive.namelist()) != len(hashes) + 1 or set(archive.namelist()) != {*hashes, "bundle_manifest.json"}:
        raise ValueError("Unexpected or duplicate bundle entries")
    verified = {}
    for name, expected in hashes.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name:
            raise ValueError("Unsafe archive path")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f"Corrupt source file: {name}")
        target = REPO / relative
        if target.is_symlink() or not target.resolve().is_relative_to(REPO.resolve()):
            raise ValueError("Extraction would leave the source directory")
        if target.exists() and target.read_bytes() != data:
            raise ValueError(f"Extracted code changed: {target}; do not alter an active run")
        verified[target] = data
for target, data in verified.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_bytes(data)
print(f"Verified {len(hashes)} files. Code snapshot: {EXPECTED_CODE_SHA256}")
print(f"Source: {REPO}")
print(f"{PHASE}: {DATASET}, {POOL_SIZE} initial questions, {STRATEGY}")
print("No experiments have run in this cell.")

Verified 73 files. Code snapshot: 1a48d14f60f807b1bd980201b84f5c86aacdb876d72bc7dcbcf8e0e26e2f6ab8
Source: /workspace/agent-repair-iclr2027/code/1a48d14f60f8
pilot: hotpotqa, 10 initial questions, unc__perplexity__argmax__bt2
No experiments have run in this cell.


## 2. GPU and Environment Preflight

The worksheet leaves $20 uncommitted and at most $99 for all compute,
including installation, downloads, idle time, development and test execution.
A session allowance applies from the provider's billing start, not from this
cell. Subtract time already billed when arranging the provider stop. Enter
gross usage since allocating $119, before credits, including other account
workloads consuming the same credit pool. A $0 invoice after credits does
not mean $0 compute use. The notebook cannot read your AWS account.
Confirm the current rate and provider controls before enabling GPU work.

A separate Python environment keeps vLLM out of the notebook kernel. The
entry-point version is pinned to 0.19.0; resolved dependencies are recorded
and checked on resume. This is not a claimed reproduction of the old environment.
The installed driver must support the selected wheel stack. No GPU model is
silently replaced with a smaller one if the machine is unsuitable.

On dependency errors, inspect the complete exception and the
[vLLM installation guide](https://docs.vllm.ai/en/v0.19.0/getting_started/installation/gpu/).
Do not layer incompatible CUDA/PyTorch wheels into an existing environment.

In [3]:
import platform
import runpy
import shutil
import subprocess

budget_allocation = runpy.run_path(str(REPO / "scripts/estimate_budget.py"))["budget_allocation"]
BUDGET = None
if GPU_HOURLY_USD is None:
    if EXECUTE_GPU:
        raise ValueError("Enter the live quote in GPU_HOURLY_USD before GPU execution")
    print("QUOTE REQUIRED: no hourly cost or runtime allowance is estimated.")
else:
    BUDGET = budget_allocation(total_usd=TOTAL_BUDGET_USD, spent_usd=SPENT_SO_FAR_USD,
                               reserve_usd=NONCOMPUTE_RESERVE_USD,
                               hourly_usd=GPU_HOURLY_USD, session_usd=SESSION_ALLOWANCE_USD)
    print(json.dumps(BUDGET, indent=2))
print("Planning only. This notebook does not stop the instance or enforce cloud billing.")
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU_INDEX, PYTHONUNBUFFERED="1")
ENV["HF_HOME"] = str(STORAGE_ROOT / "huggingface")
VENV = (Path("/content/agent-repair-venv") if PLATFORM == "colab"
        else STORAGE_ROOT / "venv-vllm019")
PYTHON = VENV / "bin/python"
if EXECUTE_GPU:
    if not BILLING_RATE_CONFIRMED or not PROVIDER_BUDGET_CONTROLS_CONFIRMED:
        raise ValueError("Confirm the live quote, funding allocation and independent stop first")
    if PLATFORM == "aws" and not AWS_CREDITS_CONFIRMED:
        raise ValueError("Confirm current AWS credit eligibility, balance, expiry and other account usage")
    if not PERSISTENT_STORAGE_CONFIRMED:
        raise ValueError("Verify your persistent mount, then set PERSISTENT_STORAGE_CONFIRMED=True")
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This environment recipe targets Linux x86-64 NVIDIA instances")
    if not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("Use a Python 3.10-3.13 notebook kernel")
    subprocess.run(["nvidia-smi", "-i", GPU_INDEX, "--query-gpu=name,memory.total,driver_version",
                    "--format=csv,noheader"], check=True)
    STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(STORAGE_ROOT).free < 100 * 1024**3 and not (STORAGE_ROOT / "huggingface").exists():
        raise RuntimeError("Provision at least 100 GiB free for model cache and initial results")
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
    installed = subprocess.run([str(PYTHON), "-c", "import vllm; assert vllm.__version__ == '0.19.0'"],
                               capture_output=True, text=True)
    if installed.returncode:
        subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(REPO / "requirements.txt"),
                        "vllm==0.19.0", "pytest"], check=True)
    subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
    subprocess.run([str(PYTHON), "-c",
                    "import torch,vllm; assert torch.cuda.is_available(); "
                    "g=torch.cuda.get_device_properties(0); print(g, torch.__version__, vllm.__version__); "
                    "assert g.total_memory >= 70 * 1024**3, 'Use the intended 80GB-class GPU for this pilot'"],
                   check=True, env=ENV)
else:
    print("PLAN ONLY: GPU checks, installations, downloads and inference are disabled.")

def call_python(source, settings):
    if not EXECUTE_GPU:
        raise RuntimeError("GPU execution is disabled")
    subprocess.run([str(PYTHON), "-c", "import json,sys; params=json.load(sys.stdin)\n" + source],
                   input=json.dumps(settings), text=True, cwd=REPO, env=ENV, check=True)

{
  "total_usd": 119.0,
  "spent_usd": 5.5,
  "reserve_usd": 20.0,
  "hourly_usd": 5.84531,
  "session_usd": 4.0,
  "remaining_compute_usd": 93.5,
  "remaining_compute_hours": 15.995729909962005,
  "session_hours": 0.6843093009609413,
  "session_seconds": 2463,
  "billing_enforced": false
}
Planning only. This notebook does not stop the instance or enforce cloud billing.
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB, 595.91.07


No broken requirements found.


_CudaDeviceProperties(name='NVIDIA RTX PRO 6000 Blackwell Server Edition', major=12, minor=0, total_memory=97251MB, multi_processor_count=188, uuid=2898e357-6583-f3ec-a7e3-7d4dee5680ba, pci_bus_id=43, pci_device_id=0, pci_domain_id=0, L2_cache_size=128MB) 2.10.0+cu128 0.19.0


## 3. Run CPU Regressions Before Downloading Model Weights

In [4]:
if EXECUTE_GPU:
    subprocess.run([str(PYTHON), "-m", "pytest", "-q",
                    "tests/test_controlled_repair.py", "tests/test_reviewer_fixes.py",
                    "tests/test_reviewer_analysis.py", "tests/test_vllm_limits.py",
                    "tests/test_cloud_runs.py", "tests/test_budget.py"], cwd=REPO, check=True, env=ENV)
else:
    print("CPU suite is available in the bundle; it runs before GPU inference when execution is enabled.")

.....................................

............................        [100%]
65 passed in 0.75s


## 4. Freeze the Run and Model Snapshot

Each dataset gets its own pool, checkpoints, and outputs. For a study across
several machines, use the same storage mount path, code snapshot, resolved
model revision, environment, and study run ID. Only `DATASET` and its cohort
files differ. Never let two workers write the same dataset/run directory.
A kernel disconnect can interrupt a stage; rerun unchanged cells to resume.

**For test runs:** supply frozen disjoint IDs and a completed policy manifest.
No test sample size or selected policy is invented here. All pilot/development
IDs belong in the explored-ID exclusion list. A manifest cannot prove that
this list is complete or that selection really preceded evaluation.

In [5]:
SETTINGS = dict(dataset=DATASET, phase=PHASE, run_id=RUN_ID, strategy=STRATEGY,
                pool_size=POOL_SIZE, batch_size=BATCH_SIZE, origin_sweep=ORIGIN_SWEEP,
                include_diagnostics=INCLUDE_DIAGNOSTICS,
                multipliers=MULTIPLIERS, code_sha256=EXPECTED_CODE_SHA256,
                test_ids=TEST_IDS, explored_ids=EXPLORED_IDS, policy_file=FROZEN_POLICY)
if EXECUTE_GPU:
    if MODEL_ID != "Qwen/Qwen2.5-32B-Instruct-AWQ":
        raise ValueError("Use a separate reviewed profile for another model family")
    if PHASE == "test" and (not MODEL_REVISION or not TEST_IDS or not EXPLORED_IDS or not FROZEN_POLICY):
        raise ValueError("Supply a pinned model revision, both ID manifests, and frozen policy for test")
    call_python("""
from pathlib import Path
from src.utils.cloud_runs import run_directory, pin_model, prepare_config, write_once
import platform, subprocess
run_dir = run_directory(params['storage'], params['settings']['dataset'],
                        params['settings']['phase'], params['settings']['run_id'])
packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines()
write_once(run_dir / 'environment_lock.json', {'python': platform.python_version(), 'packages': sorted(packages)})
pin = pin_model(run_dir, Path(params['storage']) / 'huggingface/hub', params['model_id'], params['revision'])
config = prepare_config(params['repo'], run_dir, model_pin=pin, **params['settings'])
print('Model/tokenizer commit:', pin['revision'])
print('Frozen configuration:', config)
""", dict(storage=str(STORAGE_ROOT), repo=str(REPO), model_id=MODEL_ID,
  revision=MODEL_REVISION, settings=SETTINGS))
    RUN_DIR = STORAGE_ROOT / "runs/qwen32b" / PHASE / RUN_ID / DATASET
    CONFIG = RUN_DIR / "config.yaml"
    MODEL_PIN = json.loads((RUN_DIR / "model_snapshot.json").read_text())
    print(json.dumps(MODEL_PIN, indent=2))
else:
    print(json.dumps(SETTINGS, indent=2))
    print("No run config, model snapshot or results have been created.")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Fetching 15 files:  53%|█████▎    | 8/15 [00:00<00:00, 19.77it/s]

Fetching 15 files:  67%|██████▋   | 10/15 [00:20<00:10,  2.00s/it]

Fetching 15 files:  73%|███████▎  | 11/15 [02:29<00:54, 13.55s/it]

Fetching 15 files:  80%|████████  | 12/15 [02:29<00:37, 12.48s/it]

Fetching 15 files:  87%|████████▋ | 13/15 [02:34<00:23, 11.88s/it]

Fetching 15 files:  93%|█████████▎| 14/15 [02:37<00:11, 11.27s/it]

Model/tokenizer commit: 5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c
Frozen configuration: /workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/config.yaml
{
  "repo_id": "Qwen/Qwen2.5-32B-Instruct-AWQ",
  "revision": "5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c",
  "snapshot_path": "/workspace/agent-repair-iclr2027/huggingface/hub/models--Qwen--Qwen2.5-32B-Instruct-AWQ/snapshots/5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c"
}


Fetching 15 files: 100%|██████████| 15/15 [02:38<00:00, 10.54s/it]


## 5. Generate Initial Trajectories and Score Stored Log Probabilities

These stages use one model process at a time. The process exits between
stages to release GPU memory. Stored-token uncertainty needs no sampling
calls. There is no 72B judge in the primary gold-free comparison.
If a download or inference stage fails, later stages must not be run until
it is resolved. Preserve the failed attempt and its timing record.

In [6]:
def stage(script, *extra):
    call_python("""
from src.utils.cloud_runs import run_stage
run_stage(sys.executable, params['repo'], params['run_dir'], params['script'], params['config'], params['extra'])
""", dict(repo=str(REPO), run_dir=str(RUN_DIR), script=script,
  config=str(CONFIG), extra=list(extra)))

if EXECUTE_GPU:
    stage("run_setup.py")
    stage("run_generate.py")
    stage("run_uncertainty.py")
else:
    print("PLAN ONLY: setup -> initial generation -> stored-token uncertainty.")

22:25:31 | INFO    | setup | config=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/config.yaml  base=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2
22:25:31 | INFO    | setup | Dataset: hotpotqa (file: hotpot_dev_distractor_v1.json)
Trying direct: http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json


  failed: <urlopen error timed out>
Trying direct: https://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json


  failed: <urlopen error timed out>
22:26:11 | INFO    | setup | Direct download unavailable -> Hugging Face fallback


Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating train split:  17%|█▋        | 15000/90447 [00:00<00:01, 60771.32 examples/s]

...

22:29:53 | INFO    | stage1 | 8/10  (0.18 q/s, ETA 0 min)


22:30:00 | INFO    | stage1 | 10/10  (0.19 q/s, ETA 0 min)
22:30:00 | INFO    | stage1 | TOTAL 10 | success 6 (60.0%) | FAILED 4
22:30:00 | INFO    | stage1 | Wrote failed_ids.json — input for Stages 2-5.
(EngineCore pid=3048) INFO 09-10 22:30:00 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=3048) INFO 09-10 22:30:00 [core.py:1233] Shutdown complete


(EngineCore pid=3048) Traceback (most recent call last):
(EngineCore pid=3048)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=3048)     self.run()
(EngineCore pid=3048)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=3048)     self._target(*self._args, **self._kwargs)
(EngineCore pid=3048)   File "/workspace/agent-repair-iclr2027/venv-vllm019/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1101, in run_engine_core
(EngineCore pid=3048)     engine_core.run_busy_loop()
(EngineCore pid=3048)   File "/workspace/agent-repair-iclr2027/venv-vllm019/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1144, in run_busy_loop
(EngineCore pid=3048)     raise SystemExit
(EngineCore pid=3048) SystemExit
(EngineCore pid=3048) 
(EngineCore pid=3048) During handling of the above exception, another exception occurred:
(EngineCore pid=3048) 
(EngineCore pid=3048) Traceback (most recent call last)

/usr/lib/python3.12/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
/usr/lib/python3.12/multiprocessing/resource_tracker.py:267: UserWarning: resource_tracker: '/loky-3048-1yp2vxrq': [Errno 2] No such file or directory
  warnings.warn('resource_tracker: %r: %s' % (name, e))


22:30:01 | INFO    | stage2 | config=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/config.yaml  base=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2
22:30:01 | INFO    | stage2 | Math metrics: 10 of 10 trajectories to do
22:30:01 | INFO    | stage2 | Math metrics done.
22:30:01 | INFO    | stage2 | Sampling metrics disabled; no model calls required.


## 6. Inspect the Repair Count, Then Execute

Four conditions, three seeds and one token budget require at most
`12 * N_failed` unique repair jobs. The ten-question pilot therefore has
at most 120 repairs. Policies sharing an effective origin/prompt reuse one
execution. The previous eight-origin/two-budget profile allowed 48 jobs
per failed question; the new upper bound is 75% smaller, not a measured
runtime saving. No all-origin curve or offset ablation is produced here.
Keep all three seeds and freeze the affordable question cohort before test
outcomes. An interrupted partial cohort is not a completed comparison.

In [7]:
if EXECUTE_GPU:
    stage("run_repair.py", "--dry-run")
    call_python("""
from pathlib import Path
from src.utils import load_config, save_json
from src.utils.cloud_runs import repair_plan
plan = repair_plan(load_config(params['config']))
save_json(plan, Path(params['run_dir']) / 'repair_plan.json')
print(json.dumps(plan, indent=2))
if plan['failed_questions'] == 0:
    raise ValueError('No failures in this cohort; preserve the pilot rather than fabricating repair trials')
if plan['missing_executions'] > params['maximum']:
    raise ValueError('Repair count exceeds the declared guard; inspect the plan and budget before proceeding')
""", dict(config=str(CONFIG), run_dir=str(RUN_DIR), maximum=MAX_REPAIR_EXECUTIONS))
else:
    print(f"PLAN ONLY: maximum new repair jobs allowed = {MAX_REPAIR_EXECUTIONS}")

22:30:02 | INFO    | stage5 | config=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/config.yaml  base=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2
22:30:02 | INFO    | stage5 | DRY RUN: 25 unique planned repairs, 48 strategy rows, 0 new executions; step mode=new
{
  "failed_questions": 4,
  "planned_unique_executions": 25,
  "strategy_rows": 48,
  "cached_executions": 0,
  "missing_executions": 25,
  "per_question_execution_upper_bound": 12
}


In [8]:
if EXECUTE_GPU:
    # Recheck the guard so executing this cell alone cannot bypass it.
    plan = json.loads((RUN_DIR / "repair_plan.json").read_text())
    if plan["missing_executions"] > MAX_REPAIR_EXECUTIONS:
        raise RuntimeError("Review the repair-count guard before running")
    stage("run_repair.py")
else:
    print("PLAN ONLY: repair generation is disabled.")

22:30:02 | INFO    | stage5 | config=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/config.yaml  base=/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2


22:30:03 | INFO    | stage5 | GPU VRAM: 101.975851008 GB | agent model: /workspace/agent-repair-iclr2027/huggingface/hub/models--Qwen--Qwen2.5-32B-Instruct-AWQ/snapshots/5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c (exact configured model; no automatic model or quantization fallback)


INFO 09-10 22:30:07 [utils.py:233] non-default args: {'trust_remote_code': True, 'seed': 42, 'max_model_len': 8192, 'disable_log_stats': True, 'model': '/workspace/agent-repair-iclr2027/huggingface/hub/models--Qwen--Qwen2.5-32B-Instruct-AWQ/snapshots/5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c'}
INFO 09-10 22:30:07 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 09-10 22:30:07 [model.py:1678] Using max model len 8192


INFO 09-10 22:30:07 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 09-10 22:30:07 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-10 22:30:07 [vllm.py:790] Asynchronous scheduling is enabled.


WARNING 09-10 22:30:08 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:03<00:00, 14.81it/s]


(EngineCore pid=3434) INFO 09-10 22:30:45 [gpu_model_runner.py:6046] Graph capturing finished in 8 secs, took 0.73 GiB
(EngineCore pid=3434) INFO 09-10 22:30:45 [gpu_worker.py:597] CUDA graph pool memory: 0.73 GiB (actual), 0.84 GiB (estimated), difference: 0.11 GiB (15.1%).
(EngineCore pid=3434) INFO 09-10 22:30:45 [core.py:283] init engine (profile, create kv cache, warmup model) took 26.61 seconds


(EngineCore pid=3434) INFO 09-10 22:30:46 [vllm.py:790] Asynchronous scheduling is enabled.


22:31:55 | INFO    | stage5 | Completed: 25 unique planned repairs, 48 strategy rows, 25 new executions; step mode=new
(EngineCore pid=3434) INFO 09-10 22:31:55 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=3434) INFO 09-10 22:31:55 [core.py:1233] Shutdown complete


[rank0]:[W910 22:31:55.142085333 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## 7. Paired Analysis and Pilot Throughput

The analysis rejects incomplete question/seed coverage against the repair
manifest. It compares the declared uncertainty policy with matched restart
and random with the same backtracking offset, not a test-selected winner.
Confidence intervals resample questions, not individual seed rows. Pilot
p-values are exploratory, not confirmatory evidence.

In [9]:
if EXECUTE_GPU:
    call_python("""
from pathlib import Path
from src.utils.cloud_runs import primary_controls
import subprocess
_, baselines = primary_controls(params['strategy'])
root = Path(params['run_dir'])
for multiplier in params['multipliers']:
    output = root / 'outputs/tables' / f'paired_m{float(multiplier):g}.json'
    subprocess.run([sys.executable, 'scripts/run_paired_analysis.py', '--results',
                    str(root / 'outputs/repairs/results.jsonl'), '--strategy', params['strategy'],
                    '--baselines', *baselines, '--seeds', '0', '1', '2', '--multiplier', str(multiplier),
                    '--output', str(output)], check=True)
    print(output)
""", dict(run_dir=str(RUN_DIR), strategy=STRATEGY, multipliers=MULTIPLIERS))
    for path in sorted((RUN_DIR / "outputs/tables").glob("paired_m*.json")):
        result = json.loads(path.read_text())
        print(path.name, PHASE.upper())
        for baseline, effect in result["primary_macro_contrasts"].items():
            print(f"  vs {baseline}: {100*effect['delta']:+.2f} pp "
                  f"(95% CI {100*effect['delta_lo']:+.2f}, {100*effect['delta_hi']:+.2f}), "
                  f"N={effect['n_questions']} questions")
    attempts = [json.loads(line) for line in (RUN_DIR / "stage_attempts.jsonl").read_text().splitlines()]
    repairs = [a for a in attempts if a["script"] == "run_repair.py" and not a["extra"]]
    elapsed = sum(a["elapsed_seconds"] for a in repairs)
    count = sum(a["new_execution_files"] for a in repairs)
    if count and elapsed:
        seconds_per_execution = elapsed / count
        print(f"Observed repair wall time: {elapsed/3600:.3f} h for {count} new executions")
        print(f"Observed seconds/execution: {seconds_per_execution:.2f}, including model startup/retries")
        if PLANNED_FAILED_QUESTIONS:
            if type(PLANNED_FAILED_QUESTIONS) is not int or PLANNED_FAILED_QUESTIONS < 1:
                raise ValueError("Planned failure count must be a positive integer")
            plan = json.loads((RUN_DIR / "repair_plan.json").read_text())
            jobs = PLANNED_FAILED_QUESTIONS * plan["per_question_execution_upper_bound"]
            hours = jobs * seconds_per_execution / 3600
            print(f"Configured-profile upper-count projection: {hours:.2f} GPU-hours for {jobs} jobs")
            if GPU_HOURLY_USD is not None:
                print(f"Repair-only cost at your quote: ${hours*GPU_HOURLY_USD:.2f}")
    print("Projection excludes setup, initial generation, judging, other baselines, storage and idle billing.")
else:
    print("No result table or runtime estimate is invented in plan-only mode.")

/workspace/agent-repair-iclr2027/runs/qwen32b/pilot/aws119-pilot-g7e-v2/hotpotqa/outputs/tables/paired_m1.json
paired_m1.json PILOT
  vs full_restart: +0.00 pp (95% CI +0.00, +0.00), N=4 questions
  vs random_step__bt2: +0.00 pp (95% CI +0.00, +0.00), N=4 questions
Observed repair wall time: 0.032 h for 25 new executions
Observed seconds/execution: 4.57, including model startup/retries
Projection excludes setup, initial generation, judging, other baselines, storage and idle billing.


## 8. Export Raw Evidence Before Stopping the Instance

This export includes the resolved configuration, IDs, model pin, environment,
trajectories, raw repair executions, analysis and timing records. It excludes
model weights, the environment directory, and credentials. Keep the matching
source-code bundle alongside it. Download and verify the archive before
terminating a pod; filesystem persistence differs by volume type.

In [10]:
if EXECUTE_GPU:
    import tarfile
    from datetime import datetime, timezone
    export_dir = STORAGE_ROOT / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    target = export_dir / f"{PHASE}-{RUN_ID}-{DATASET}-{stamp}.tar.gz"
    with tarfile.open(target, "w:gz") as archive:
        archive.add(RUN_DIR, arcname=DATASET, recursive=True)
    digest = hashlib.sha256()
    with target.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    target.with_suffix(target.suffix + ".sha256").write_text(digest.hexdigest() + "  " + target.name + "\n")
    print("Result archive:", target)
    print("SHA256:", digest.hexdigest())
    print("Download via your provider's file interface or scp; this cell does not stop billing.")
else:
    print("No experimental outputs exist to export in this plan-only run.")

Result archive: /workspace/agent-repair-iclr2027/exports/pilot-aws119-pilot-g7e-v2-hotpotqa-20260910T223157134317Z.tar.gz
SHA256: 129671d66e603c31046b579d025e6c7cdc0d1f950ea53aca8e0d0151a412c19d
Download via your provider's file interface or scp; this cell does not stop billing.


## 9. Move From the Pilot to the Study

1. Audit token/new-step allowances, prefix replay, scoring, missing-logprob rates
   and complete trial coverage on the actual GPU. Retain all pilot IDs as explored.
2. Benchmark batch sizes on development-only runs with different run IDs. Do not
   change batching mid-study: batching and the serving stack can affect outputs.
3. Select one policy on development data across QA3, choose sample sizes from
   paired-difference variance and a stated precision target, and freeze the policy
   below before any test outcomes are examined. Hash IDs with the same helper.
4. Reuse one GPU sequentially with a different `DATASET`; keep the common policy,
   model revision, run ID and mount paths. Update actual spending before each
   session. Do not fund extra datasets or model families within this profile.
   Select sample sizes using development-only throughput and uncertainty;
   never stop or extend test collection based on whether an effect looks positive.
5. Combine one completed raw file per dataset using `run_paired_analysis.py --results`
   followed by all three paths. Do not average pilot p-values or merge different models.

The core notebook does not implement the development-fitted position-matched
random baseline, diagnosis/replay comparator, repeated-restart answer selector,
or blinded human labeling. These remain study requirements where their claims
are retained. FEVER is deliberately excluded until evidence and prompting are fixed.

### Optional: Freeze the Study Policy After Development

This cell computes cohort hashes from your actual files; it does not select
a winner or invent a sample size. Complete the fields and enable the switch
only after the development selection and precision analysis are complete.
The file cannot be overwritten with different settings. `n_questions` counts
initial questions; the number of failures is observed after generation.
Point `FROZEN_POLICY` to the resulting JSON in each test notebook.

In [11]:
FREEZE_STUDY_POLICY = False
SELECTED_STRATEGY = ""  # selected using DEVELOPMENT outcomes only
SELECTION_NOTE = ""    # development records, candidates and tie-breaking rule
PRECISION_TARGET = ""  # interval-width target and sample-size rationale
STUDY_MULTIPLIERS = [1.0]
STUDY_ORIGIN_SWEEP = False
STUDY_INCLUDE_DIAGNOSTICS = False
STUDY_BATCH_SIZE = 2
COHORT_FILES = {
    "hotpotqa": {"test": None, "explored": None},
    "musique": {"test": None, "explored": None},
    "2wikimultihopqa": {"test": None, "explored": None},
}
if FREEZE_STUDY_POLICY:
    if not EXECUTE_GPU or "MODEL_PIN" not in globals():
        raise RuntimeError("Complete the environment and model-pin cells before freezing a policy")
    if PHASE == "test":
        raise ValueError("Do not create a study policy after opening test outcomes")
    if not SELECTED_STRATEGY or not SELECTION_NOTE or not PRECISION_TARGET:
        raise ValueError("Complete the development selection and precision rationale")
    call_python("""
from src.utils.cloud_runs import primary_controls, write_once
from src.utils.cohorts import read_ids
from src.repair.controlled import fingerprint
primary_controls(params['strategy'], include_diagnostics=params['include_diagnostics'])
policy = {key: params[key] for key in ('strategy', 'selection_note', 'precision_target',
           'model_id', 'model_revision', 'code_sha256', 'multipliers', 'origin_sweep',
           'include_diagnostics', 'batch_size')}
policy.update(seeds=[0, 1, 2], max_steps=8, max_tokens_per_step=512, n_questions={}, cohort_sha256={})
for dataset, paths in params['cohorts'].items():
    if not paths['test'] or not paths['explored']:
        raise ValueError('Provide both ID files for every QA dataset')
    ids, explored = read_ids(paths['test']), read_ids(paths['explored'])
    if not ids or not explored or set(ids) & set(explored):
        raise ValueError('Cohorts must be nonempty and disjoint')
    policy['n_questions'][dataset] = len(ids)
    policy['cohort_sha256'][dataset] = {'test': fingerprint(ids), 'explored': fingerprint(explored)}
write_once(params['output'], policy)
print('Frozen policy:', params['output'])
print(json.dumps(policy, indent=2))
""", dict(strategy=SELECTED_STRATEGY, selection_note=SELECTION_NOTE,
  precision_target=PRECISION_TARGET, model_id=MODEL_ID,
  model_revision=MODEL_PIN["revision"], code_sha256=EXPECTED_CODE_SHA256,
  multipliers=STUDY_MULTIPLIERS, origin_sweep=STUDY_ORIGIN_SWEEP,
  include_diagnostics=STUDY_INCLUDE_DIAGNOSTICS,
  batch_size=STUDY_BATCH_SIZE, cohorts=COHORT_FILES,
  output=str(STORAGE_ROOT / "study_policy.json")))
else:
    print("No study policy has been selected or frozen automatically.")

No study policy has been selected or frozen automatically.
